# Prompt router v3 (no GPU needed)
**What changed from the last version**
- 6 task classes: `debug`, `design`, `docs` (needs fresh info: versions, news, weather, current office holders), `explain` (programming concepts), `code` (generate, refactor, shell, SQL, regex) and **`general`** (non-programming questions: capitals, locations, facts, maths, chit-chat, long essays and plans).
- Much more varied and messy training prompts (hundreds of unique ones per class, with typos, lowercase, slang), plus real Hugging Face data.
- Both tiers exist for most classes, so tier is not tied to a single source.
- Safety rules for obvious cases (stack trace → debug, "latest / deprecated" → docs).
- Reports the accuracy that matters: did the prompt reach the right **model**?
- Saves `requirements.txt` so your laptop uses the same library versions.

Just click **Runtime → Run all**. Any Hugging Face dataset that fails prints `SKIP` and the rest still works.

In [ ]:
!pip -q install datasets sentence-transformers scikit-learn joblib fastapi uvicorn

## 1. Shared code (training and server both use it)

In [ ]:
%%writefile features.py
import re, math, numpy as np

FRESH  = re.compile(r"\b(latest|deprecated|release notes|changelog|still (supported|maintained)|new in|what changed|breaking changes?)\b|\bv?\d+\.\d+(\.\d+)?\b", re.I)
TRACE  = re.compile(r"traceback|exception|error:|stack trace|segfault|at line \d+|\bpanic\b|undefined is not|panicked", re.I)
DESIGN = re.compile(r"architecture|scal(e|ab)|trade-?offs?|system design|microservice|multi-tenant|throughput|high availability", re.I)
SHELL  = re.compile(r"\b(regex|sql|bash|shell|awk|sed|cron|dockerfile|docker|curl|git)\b", re.I)
CODEKW = re.compile(r"\b(def|class|function|import|return|const|let|var|public|void|SELECT|FROM)\b")
REFAC  = re.compile(r"\b(refactor|clean up|review|optimi[sz]e|rewrite|simplify)\b", re.I)

def hand_features(p):
    words = len(p.split())
    return [math.log1p(words), math.log1p(p.count("\n")), float("```" in p),
            float(bool(TRACE.search(p))), float(bool(FRESH.search(p))),
            float(bool(DESIGN.search(p))), float(bool(SHELL.search(p))),
            float(bool(CODEKW.search(p))), float(bool(REFAC.search(p))),
            p.count("?")/3.0]

def featurize(texts, encoder):
    texts = list(texts)
    emb = encoder.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    hand = np.array([hand_features(t) for t in texts])
    return np.hstack([emb, hand])

# ---- safety rules for obvious cases ----
STRONG_TRACE = re.compile(r"traceback \(most recent call last\)|^\s*File \".+\", line \d+|segmentation fault|segfault|\b[A-Za-z_.]*(Error|Exception):|\bpanic:|exit code \d+", re.I | re.M)
FRESH_KW = re.compile(r"\b(latest|deprecated|release notes|changelog|what changed|what'?s new|whats new|still maintained|still supported|breaking changes?|is out yet|out yet)\b", re.I)
NOW_KW = re.compile(r"\b(weather|latest news|news on|live score|score of|who won|current (president|prime minister|ceo|champion)|election results|stock price)\b", re.I)
CODE_ASK = re.compile(r"\b(write|create|implement|function|script|query|regex|command|code|build|make|convert|refactor|git|sql|bash|file|files)\b", re.I)

def apply_rules(p, task, tier):
    why = []
    if STRONG_TRACE.search(p) and task != "debug":
        task = "debug"; why.append("Rule: contains an error or stack trace, so debug")
    elif FRESH_KW.search(p) and "```" not in p and not CODE_ASK.search(p) and task != "docs":
        task = "docs"; why.append("Rule: asks about latest or deprecated things, so it needs fresh info")
    elif NOW_KW.search(p) and "```" not in p and not CODE_ASK.search(p) and task in ("general", "explain"):
        task = "docs"; why.append("Rule: needs live or current information, so search")
    if task == "debug" and STRONG_TRACE.search(p) and p.count("\n") >= 3 and tier != "expensive":
        tier = "expensive"; why.append("Rule: multi-line stack trace needs a stronger model")
    return task, tier, why

# ---- routing table (keys match MODELS in the web page) ----
ROUTES = {
    "debug":   {"expensive": "claude", "cheap": "codeS"},
    "design":  {"expensive": "reason", "cheap": "claude"},
    "code":    {"expensive": "claude", "cheap": "codeS"},
    "explain": {"expensive": "claude", "cheap": "mini"},
    "docs":    {"expensive": "search", "cheap": "search"},
    "general": {"expensive": "claude", "cheap": "mini"},
}
def route(task, tier): return ROUTES[task][tier]

## 2. Build the dataset
Real Hugging Face data (best effort) plus a large generator of varied, messy developer prompts for the categories no dataset covers.

In [ ]:
import re, os, random, pandas as pd, numpy as np
from datasets import load_dataset
SEED = 0; CAP = 300          # max rows per (task, tier)
random.seed(SEED)
pick = random.choice

# ---------- messifier: makes clean template text look like real typing ----------
SLANG = {"please":"pls","you":"u","your":"ur","with":"w/","about":"abt","function":"func","doesn't":"doesnt","don't":"dont","because":"bc","before":"b4","without":"w/o","between":"btw"}
def messify(t):
    if "\n" in t or "```" in t:
        return pick(["", "", "hey ", "ok so ", "bro ", "help: "]) + t
    if random.random() < 0.35: return t
    if random.random() < 0.6: t = t.lower()
    t = " ".join(SLANG.get(w.lower(), w) if random.random() < 0.5 else w for w in t.split())
    if random.random() < 0.5: t = t.rstrip("?.!")
    return pick(["", "", "hey ", "ok so ", "bro ", "urgent: ", "quick q: ", "pls help "]) + t + pick(["", "", "", " pls", " thanks", " asap", " ??"])

# ---------- shared pools ----------
LANGS = ["python","javascript","typescript","java","go","rust","c++","c#","php","ruby","kotlin","swift"]
FWS = ["flask","fastapi","react","vue","angular","express","spring boot","laravel","rails","node","pandas","numpy","pytorch","docker","kubernetes","postgres","mongodb","redis","nginx","terraform","aws lambda","svelte","prisma","sqlalchemy","webpack","jest"]
ERRS = ["KeyError: 'user_id'", "TypeError: 'NoneType' object is not subscriptable", "ValueError: invalid literal for int() with base 10", "IndexError: list index out of range",
        "AttributeError: 'str' object has no attribute 'decode'", "ImportError: cannot import name 'soft_unicode'", "ModuleNotFoundError: No module named 'requests'",
        "TypeError: Cannot read properties of undefined (reading 'map')", "ReferenceError: window is not defined", "SyntaxError: Unexpected token '<'", "Uncaught TypeError: x is not a function",
        "java.lang.NullPointerException", "java.lang.OutOfMemoryError: Java heap space", "ClassNotFoundException: com.mysql.jdbc.Driver", "panic: runtime error: index out of range [3] with length 3",
        "fatal error: all goroutines are asleep - deadlock!", "error[E0382]: borrow of moved value", "CORS error: No 'Access-Control-Allow-Origin' header present", "ECONNREFUSED 127.0.0.1:5432",
        "ENOSPC: no space left on device", "Permission denied (publickey)", "CrashLoopBackOff", "OOMKilled", "UnicodeDecodeError: 'utf-8' codec can't decode byte 0xff", "RecursionError: maximum recursion depth exceeded",
        "ORA-00942: table or view does not exist", "psycopg2.OperationalError: connection refused", "429 Too Many Requests", "SSL: CERTIFICATE_VERIFY_FAILED", "Segmentation fault (core dumped)"]
FILES = ["app.py","views.py","index.js","UserService.java","main.go","App.jsx","worker.py","server.ts","etl.py","handlers.rb","models.py","api.ts","Controller.php"]
FUNCS = ["handle","process","run","load","render","parse","fetch_user","save","validate","merge","dispatch","init"]

# ---------- DEBUG ----------
def debug_cheap():
    e, f = pick(ERRS), pick(FWS)
    return messify(pick([f"what does '{e}' mean", f"getting {e} in {f}", f"{e} fix?", f"{f} {e} what is going on", f"why does {f} give {e}", f"how do i fix {e}",
                         f"{f} wont start after install", f"my {f} build keeps failing with {e}", f"{e} only on my machine", f"is {e} a {f} bug or my code"])), "cheap"
def debug_exp():
    f, fn, n, e = pick(FILES), pick(FUNCS), random.randint(8, 500), pick(ERRS)
    if random.random() < 0.5:
        tb = f'Traceback (most recent call last):\n  File "{f}", line {n}, in {fn}\n    result = {pick(FUNCS)}(data)\n  File "{pick(FILES)}", line {random.randint(5,300)}, in {pick(FUNCS)}\n    {e}'
        return messify(tb + "\n" + pick(["why does this happen intermittently under load?", "only fails in production, not locally", "how do i fix this?", "what is the root cause here", "happens about once a day", "it started after i upgraded dependencies"])), "expensive"
    fw = pick(FWS)
    sym = pick(["crashes","hangs","leaks memory","returns stale data","is randomly slow","deadlocks","drops requests","double processes jobs"])
    ctx = pick(["only in production","under load","after upgrading","on the second request","when two users hit it at once","only in ci","after about an hour","with large payloads"])
    det = pick(["i cant reproduce it locally", "logs show nothing useful", "heap snapshots show growth but no clear owner", "retries make it worse", "it started last week with no deploy", "two workers share a lock"])
    return messify(pick([f"my {fw} service {sym} {ctx}, {det}", f"{fw} app {sym} {ctx}. {det}. where do i even start", f"debugging a {fw} issue: it {sym} {ctx} and {det}"])), "expensive"

# ---------- DESIGN ----------
SYSTEMS = ["multi-tenant job queue","real-time chat backend","rate limiter for a public api","url shortener","notification service","payments ledger","feature flag service","log ingestion pipeline",
           "video upload and transcoding service","search autocomplete","ride matching service","inventory system for a marketplace","ticket booking system","analytics dashboard backend","file sync service","recommendation feed","ci build farm","audit logging system"]
REQS = ["scales to 10k requests per second","survives a regional outage","needs exactly-once processing","serves 5M daily users","stays under 100ms p99 latency","has strict compliance rules","must support offline clients","needs zero downtime deploys","handles bursty traffic 50x normal","spans three teams"]
TECH = ["kafka","sqs","rabbitmq","postgres","mongodb","dynamodb","redis","memcached","rest","graphql","grpc","a monolith","microservices","serverless","kubernetes","a vm","websockets","polling","sqlite","elasticsearch"]
SMALL = ["side project","todo app","personal blog","small internal tool","weekend hackathon app","tiny saas","portfolio site","discord bot"]
def design_exp():
    s, r, a, b = pick(SYSTEMS), pick(REQS), pick(TECH), pick(TECH)
    return messify(pick([f"Design a {s} that {r}. Compare {a} vs {b} and explain the tradeoffs.", f"how should i architect a {s}? it {r}", f"system design: {s}, it {r}. what are the failure modes and how do we migrate later",
                         f"we need a {s} that {r}. walk me through the architecture, data model and scaling plan", f"{s} design review: {a} or {b}, and what breaks first when it {r}"])), "expensive"
def design_cheap():
    a, b, sm = pick(TECH), pick(TECH), pick(SMALL)
    return messify(pick([f"should i use {a} or {b} for a {sm}", f"is {a} overkill for a {sm}", f"{a} vs {b} for a {sm}, which one", f"{a} or {b} for a {sm}?", f"do i need {a} for a {sm}"])), "cheap"

# ---------- DOCS (needs fresh info -> search model) ----------
LIBS = ["react","vue","svelte","fastapi","spring boot","node.js","typescript","vite","pytorch","numpy","rust","go","kubernetes","terraform","express","angular","flutter","prisma","sqlalchemy","webpack","jest","eslint","deno","bun","htmx","astro","remix","nuxt","supabase","openai sdk","docker compose","java","kotlin","swift","pandas","polars"]
FEATS = ["server components","the old config format","commonjs","async handlers","type hints","hot reload","the legacy api","decorators","ssr","websocket support","the cli flag --watch","windows support"]
def docs_q():
    l, l2, ft = pick(LIBS), pick(LIBS), pick(FEATS)
    v, v2 = f"{random.randint(1,19)}.{random.randint(0,9)}", f"{random.randint(1,19)}.{random.randint(0,9)}"
    return messify(pick([f"what's new in {l} {v}", f"latest {l} version", f"is {l} {v} still supported", f"{l} {v} breaking changes", f"{l} changelog", f"does {l} support {ft} now", f"is {ft} deprecated in {l}",
                         f"migrating {l} {v} to {v2}, anything that breaks", f"current recommended way to set up {l} in 2026", f"is {l} dead or still maintained", f"{l} vs {l2} in 2026, which is more popular now",
                         f"any security advisory for {l} {v}", f"{l} release notes {v}", f"did {l} drop {ft} in the new release"])), "cheap"

# ---------- EXPLAIN ----------
CONCEPTS = ["a decorator","async await","dependency injection","a mutex","cors","a docker image","a hash map","a b-tree","oauth","jwt","a race condition","idempotency","eventual consistency","a monad","tail recursion","garbage collection",
            "a reverse proxy","a cdn","websockets","a deadlock","polymorphism","a pointer","a generator","a context manager","the event loop","a primary key","an index in a database","normalization","a transaction","rate limiting",
            "a bloom filter","a trie","dns","tcp vs udp","https","a thread pool","a semaphore","immutability","currying","a linked list","a binary tree","bitwise operators","a webhook","ci cd","a cron job","unit vs integration tests"]
def explain_cheap():
    c, c2 = pick(CONCEPTS), pick(CONCEPTS)
    return messify(pick([f"what is {c}", f"explain {c} simply", f"{c} meaning?", f"eli5 {c}", f"hey what does {c} do", f"difference between {c} and {c2}", f"can u explain {c} real quick", f"when would i use {c}", f"is {c} the same as {c2}"])), "cheap"
def explain_exp():
    c = pick(CONCEPTS)
    return messify(pick([f"explain {c} in depth: how it works internally, tradeoffs, edge cases and failure modes", f"walk me through how {c} really works under the hood, how it breaks, and how to debug it in production",
                         f"i want a deep dive on {c}: internals, performance implications, concurrency issues and real world pitfalls", f"compare {c} with {pick(CONCEPTS)} in detail, covering internals, when each fails and migration costs"])), "expensive"


# ---------- GENERAL (non-programming questions) + fresh-info questions ----------
COUNTRIES = [("france","paris"),("germany","berlin"),("italy","rome"),("spain","madrid"),("japan","tokyo"),("india","new delhi"),("egypt","cairo"),("argentina","buenos aires"),("turkey","ankara"),("sweden","stockholm"),
             ("norway","oslo"),("portugal","lisbon"),("greece","athens"),("thailand","bangkok"),("vietnam","hanoi"),("south korea","seoul"),("mexico","mexico city"),("peru","lima"),("chile","santiago"),("poland","warsaw"),
             ("austria","vienna"),("ireland","dublin"),("nigeria","abuja"),("morocco","rabat"),("indonesia","jakarta"),("russia","moscow"),("china","beijing"),("united kingdom","london"),("netherlands","amsterdam"),("switzerland","bern")]
CITIES = [("paris","france"),("berlin","germany"),("rome","italy"),("tokyo","japan"),("mumbai","india"),("cairo","egypt"),("lagos","nigeria"),("istanbul","turkey"),("barcelona","spain"),("kyoto","japan"),("munich","germany"),
          ("lyon","france"),("marseille","france"),("osaka","japan"),("jaipur","india"),("casablanca","morocco"),("florence","italy"),("zurich","switzerland"),("krakow","poland"),("lima","peru"),("hanoi","vietnam")]
LANDMARKS = [("eiffel tower","paris, france"),("colosseum","rome, italy"),("taj mahal","agra, india"),("great pyramid of giza","giza, egypt"),("big ben","london, england"),("machu picchu","peru"),("louvre museum","paris, france"),
             ("statue of liberty","new york, usa"),("mount fuji","japan"),("sagrada familia","barcelona, spain"),("acropolis","athens, greece"),("brandenburg gate","berlin, germany"),("angkor wat","cambodia"),("petra","jordan"),("golden gate bridge","san francisco, usa")]
FACTQ = ["who wrote hamlet","how many planets are in the solar system","what is the boiling point of water","who discovered penicillin","what year did world war 2 end","what is the largest ocean","how many continents are there",
         "what is photosynthesis","who was the first person on the moon","what is the tallest mountain in the world","how far is the moon from earth","what is the speed of light","who invented the telephone","what is the chemical symbol for gold",
         "how many days are in a leap year","what is the largest planet","who wrote harry potter","what is the longest river in the world","who was the first president of the usa","what is the smallest prime number","what causes rainbows","how many bones are in the human body",
         "what is the capital city of the european union","who composed the four seasons","what is the freezing point of water in fahrenheit","what gas do plants absorb","who discovered gravity","how many players are on a football team","what is the hottest planet","which animal is the largest on earth"]
CHAT = ["hi","hello","hey there","how are you","hi how are you doing","good morning","thanks a lot","tell me a joke","who are you","what can you do","how is it going","im bored","can you help me","thank you","what's up","good night","lol ok","tell me something interesting"]
def general_cheap():
    c, cap = pick(COUNTRIES); city, cc = pick(CITIES); lm, lloc = pick(LANDMARKS)
    k = random.random()
    if k < 0.28: t = pick([f"what is the capital of {c}", f"capital of {c}", f"{c} capital?", f"which city is the capital of {c}", f"tell me the capital city of {c}", f"whats the capital of {c}", f"capital city of {c} and where is it located"])
    elif k < 0.5: t = pick([f"where is {city} located", f"which country is {city} in", f"where is {lm} located", f"where can i find the {lm}", f"{lm} location", f"where is {cap} situated", f"where is {cap} found on the map", f"in which country is the {lm}", f"{city} is in which country"])
    elif k < 0.62: t = pick([f"what currency does {c} use", f"what language is spoken in {c}", f"what continent is {c} in", f"what is {c} famous for", f"is {c} in europe"])
    elif k < 0.78: t = pick(FACTQ)
    elif k < 0.88: a, b = random.randint(2, 99), random.randint(10, 900); t = pick([f"what is {a} percent of {b}", f"convert {b} km to miles", f"what is {a} times {b}", f"whats {b} divided by {a}", f"how many minutes in {a} hours"])
    else: t = pick(CHAT)
    return messify(t), "cheap"
ESSAY = ["the causes of the industrial revolution","the economic impact of the printing press","how climate change affects coastal cities","the rise and fall of the roman empire","the ethics of gene editing","how central banks control inflation","the origins of the cold war","the history of the silk road","how vaccines changed public health","the role of social media in modern politics"]
POLICY = ["a four day work week","universal basic income","nuclear power","remote first companies","a carbon tax","free public transport","school uniforms","raising the retirement age"]
SKILLS = ["spanish","calculus","public speaking","guitar","statistics","chess","academic writing","personal finance"]
def general_exp():
    t, t2, pol, c, n = pick(ESSAY), pick(ESSAY), pick(POLICY), pick(COUNTRIES)[0], random.randint(5, 14)
    return messify(pick([f"write a detailed {random.choice([800,1000,1500])} word essay on {t}", f"compare and contrast {t} with {t2} in depth, with evidence and counterarguments",
                         f"analyze the pros and cons of {pol} and give a reasoned recommendation with numbers", f"plan a {n} day trip through {c} on a budget with transport, daily costs, and backup plans",
                         f"make a detailed {n} week study plan to learn {pick(SKILLS)} with weekly goals, resources and checkpoints", f"explain {t} in depth, covering causes, consequences, competing historians and lasting effects"])), "expensive"
SPORTS = ["ipl","champions league","wimbledon","world cup","nba finals","premier league","formula 1 race","olympic"]
TOPICS = ["ai regulation","the stock market","space launches","electric cars","the football transfer window","inflation","the monsoon","tech layoffs"]
ROLES = ["president","prime minister","ceo of apple","ceo of google","champion","chancellor"]
ASSETS = ["bitcoin","gold","the dollar against the rupee","crude oil","tesla stock"]
def docs_fresh():
    city, _ = pick(CITIES); c = pick(COUNTRIES)[0]
    return messify(pick([f"weather in {city} today", f"what is the weather like in {city} right now", f"who won the {pick(SPORTS)} yesterday", f"latest news on {pick(TOPICS)}", f"who is the current {pick(ROLES)} of {c}",
                         f"live score {pick(SPORTS)}", f"price of {pick(ASSETS)} today", f"is it going to rain in {city} tomorrow", f"{c} election results", f"what happened in {pick(TOPICS)} this week",
                         f"current {pick(ROLES)} of {c}", f"news today about {pick(TOPICS)}"])), "cheap"

# ---------- CODE (generate, shell, sql, regex, refactor) ----------
CHEAP_TASKS = ["reverses a string","checks if a number is prime","sorts a list of dicts by a key","removes duplicates from a list","counts word frequency in a text","flattens a nested list","merges two sorted arrays",
               "converts celsius to fahrenheit","finds the second largest number","validates an email address","reads a csv into a list of rows","generates a random password","computes fibonacci numbers","checks if two strings are anagrams","parses a date string","debounces a callback","deep clones an object","formats a number as currency"]
HEAVY_THINGS = ["rest api","cli tool","web scraper","job scheduler","in-memory cache","rate limiter","markdown parser","websocket server","etl pipeline","retry decorator","file watcher","plugin system","task queue","config loader"]
HEAVY_REQS = ["supports pagination","retries with exponential backoff","logs to a rotating file","has unit tests","handles concurrent requests","validates input with a schema","reads config from env vars","is thread safe","streams large files without loading them into memory","writes results to postgres","has a dry run mode","supports graceful shutdown","exposes prometheus metrics"]
SHELL_T = ["find files larger than 100mb","rename all .txt files to .md","undo the last git commit","count lines in all python files","delete merged git branches","compress a folder into tar.gz","show the 10 biggest folders","find which process uses a port","tail a log and grep errors","copy a file over ssh","list docker containers sorted by memory","change file permissions recursively","replace text in many files","show disk usage by directory","stash only some files in git"]
REGEX_T = ["a hex color","a url with optional port","ipv4 addresses","a us zip code","a semver version","dates like 2024-03-15","a username with 3 to 16 letters","html tags","a credit card number with spaces","words starting with a capital letter"]
SQL_T = ["get the 5 most recent orders per customer","find duplicate emails in a users table","count signups per month","delete rows older than 90 days","join orders and customers and sum totals","find customers with no orders","update prices by 10 percent for one category","get the median salary","list tables larger than 1gb in postgres","pivot sales by quarter"]
SNIPS = [("python","def f(a):\n    r = []\n    for x in a:\n        if x not in r:\n            r.append(x)\n    return r"),
         ("python","def g(d):\n    if d:\n        if 'k' in d:\n            if d['k']:\n                return d['k']\n    return None"),
         ("javascript","var out = [];\nfor (var i = 0; i < arr.length; i++) {\n  if (arr[i] > 10) out.push(arr[i] * 2);\n}"),
         ("python","def process(items):\n    total = 0\n    for i in items:\n        try:\n            total += int(i)\n        except:\n            pass\n    log = open('log.txt', 'a')\n    log.write(str(total))\n    return total"),
         ("javascript","function load(id, cb) {\n  getUser(id, function(u) {\n    getOrders(u.id, function(o) {\n      getItems(o[0].id, function(items) {\n        cb(items);\n      });\n    });\n  });\n}"),
         ("go","func read(p string) []byte {\n\tf, _ := os.Open(p)\n\tb, _ := io.ReadAll(f)\n\treturn b\n}")]
def code_cheap():
    k = random.random(); lang = pick(LANGS)
    if k < 0.3: t = f"{pick(['write','create','make','give me','code'])} a {lang} function that {pick(CHEAP_TASKS)}"
    elif k < 0.5: t = pick([f"command to {pick(SHELL_T)}", f"bash one-liner to {pick(SHELL_T)}", f"how do i {pick(SHELL_T)}"])
    elif k < 0.65: t = pick([f"regex for {pick(REGEX_T)}", f"regex to match {pick(REGEX_T)}"])
    elif k < 0.8: t = pick([f"sql to {pick(SQL_T)}", f"sql query: {pick(SQL_T)}"])
    else:
        l, s = pick([x for x in SNIPS if x[1].count("\n") <= 5])
        t = f"{pick(['simplify this','make this cleaner','convert this to a list comprehension','add comments to this','rename the variables in this'])}\n```{l}\n{s}\n```"
    return messify(t), "cheap"
def code_exp():
    k = random.random(); lang = pick(LANGS)
    if k < 0.6:
        reqs = random.sample(HEAVY_REQS, 3)
        t = f"{pick(['write','build','implement','create'])} a {lang} {pick(HEAVY_THINGS)} that {reqs[0]}, {reqs[1]} and {reqs[2]}"
    elif k < 0.85:
        l, s = pick([x for x in SNIPS if x[1].count("\n") >= 5])
        t = f"{pick(['refactor this and explain what you changed','review this code for bugs and performance','rewrite this to be async and add tests','clean this up, handle errors properly and add type hints'])}\n```{l}\n{s}\n```"
    else:
        t = f"{pick(['write unit tests','add integration tests','write a migration plan'])} for a {lang} {pick(HEAVY_THINGS)} that {random.choice(HEAVY_REQS)} and {random.choice(HEAVY_REQS)}, covering edge cases and failure paths"
    return messify(t), "expensive"

# ---------- generate unique prompts ----------
rows = []   # (text, task, tier, source)
def fill(fn, task, n, src="synthetic"):
    seen, tries = set(), 0
    while len(seen) < n and tries < n * 30:
        t, tier = fn(); tries += 1
        key = re.sub(r"\s+", " ", t.lower()).strip()
        if key not in seen: seen.add(key); rows.append((t, task, tier, src))
    print(f"{task:8s} {fn.__name__:12s} {len(seen)} unique")
for fn, task, n in [(debug_cheap,"debug",300),(debug_exp,"debug",300),(design_cheap,"design",250),(design_exp,"design",300),(docs_q,"docs",300),
                    (explain_cheap,"explain",300),(explain_exp,"explain",200),(code_cheap,"code",260),(code_exp,"code",260),
                    (general_cheap,"general",400),(general_exp,"general",220),(docs_fresh,"docs",250)]:
    fill(fn, task, n)

# ---------- real datasets (best effort) ----------
def add(texts, task, tier_fn, src, n):
    texts = [t.strip() for t in texts if t and 8 < len(t.strip()) < 2500]
    random.shuffle(texts)
    for t in texts[:n]: rows.append((t, task, tier_fn(t), src))
def safe(fn):
    try: fn(); print("ok  ", fn.__name__)
    except Exception as e: print("SKIP", fn.__name__, "->", str(e)[:110])

def mbpp():
    try: d = load_dataset("google-research-datasets/mbpp", "full")
    except Exception: d = load_dataset("mbpp", "full")
    add([t for s in d for t in d[s]["text"]], "code", lambda t: "cheap", "mbpp", 150)
def codealpaca():
    d = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
    add([r["instruction"] + (("\n" + r["input"]) if r.get("input") else "") for r in d.select(range(min(6000, len(d))))], "code", lambda t: "cheap" if len(t.split()) < 35 else "expensive", "codealpaca", 150)
def magicoder():
    d = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K", split="train").shuffle(seed=SEED).select(range(1000))
    add(d["problem"], "code", lambda t: "expensive", "magicoder", 150)
def evol():
    d = load_dataset("nickrosh/Evol-Instruct-Code-80k-v1", split="train").shuffle(seed=SEED).select(range(1000))
    add(d["instruction"], "code", lambda t: "expensive", "evol-instruct-code", 100)
def sqlctx():
    d = load_dataset("b-mc2/sql-create-context", split="train").shuffle(seed=SEED).select(range(600))
    add(d["question"], "code", lambda t: "cheap", "sql-create-context", 100)
def dolly():
    d = load_dataset("databricks/databricks-dolly-15k", split="train")
    add([r["instruction"] for r in d if r["category"] in ("open_qa","general_qa") and len(r["instruction"].split()) < 25], "explain", lambda t: "cheap", "dolly", 80)
for f in (mbpp, codealpaca, magicoder, evol, sqlctx, dolly): safe(f)

# ---------- your own hand-written TRAINING prompts (optional) ----------
# upload my_prompts.csv with columns: text,task,tier   (task must be one of debug, design, docs, explain, code, general)
if os.path.exists("my_prompts.csv"):
    mp = pd.read_csv("my_prompts.csv")
    for t, k, r in zip(mp.text, mp.task, mp.tier): rows.append((str(t), k, r, "hand-written"))
    print("loaded", len(mp), "hand-written prompts")

df = pd.DataFrame(rows, columns=["text","task","tier","source"])
df["norm"] = df.text.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
df = df.drop_duplicates("norm").reset_index(drop=True)
print("\n", len(df), "rows after dedupe")
print(df.groupby(["task","tier"]).size().unstack(fill_value=0))

## 3. Your TEST prompts (never trained on)
These 51 were written separately from the generator. **Add your own messy prompts at the bottom** (label format: `(prompt, task, tier)`). Tasks: `debug, design, docs, explain, code, general`. Tiers: `cheap, expensive`.

In [ ]:
MY_TEST = [
 # debug
 ("my flask app throws 500 on POST /login but works fine in postman, log says KeyError 'csrf_token'", "debug", "cheap"),
 ("getting 'cannot read properties of undefined' in react after i updated to 19, only on the dashboard page", "debug", "cheap"),
 ("this thing deadlocks about once a day in prod and i cant reproduce it locally, two workers share a redis lock\n```python\nwith lock:\n    update(db)\n    cache.set(k, v)\n```", "debug", "expensive"),
 ("pytest passes locally but fails on github actions with ModuleNotFoundError, same python version", "debug", "cheap"),
 ("memory keeps climbing in my node service after each request, heap snapshots show lots of closures, no idea where", "debug", "expensive"),
 ("docker build fails: failed to solve: process did not complete successfully: exit code 1", "debug", "cheap"),
 ("Traceback (most recent call last):\n  File \"etl.py\", line 88, in run\n    df = merge(a, b)\n  File \"etl.py\", line 41, in merge\n    return a.join(b, on='id')\nKeyError: 'id'\nit only breaks when the input file has more than 1M rows", "debug", "expensive"),
 ("segfault when i call my c++ lib from python via ctypes, works with small arrays", "debug", "expensive"),
 # design
 ("how would you design a collaborative editor like google docs, conflict resolution, offline mode", "design", "expensive"),
 ("should i use postgres or mongo for a small todo app", "design", "cheap"),
 ("we have a monolith with 40 devs, thinking about splitting into services, how do we decide boundaries and migrate without downtime", "design", "expensive"),
 ("is redis overkill for caching in a tiny flask app", "design", "cheap"),
 ("design a notification system that sends email sms push to 2m users, retries, dedupe, user preferences", "design", "expensive"),
 ("rest vs graphql for my side project api, which one?", "design", "cheap"),
 ("architecture for a multi region payments service that cant lose a transaction", "design", "expensive"),
 ("kafka or just a postgres queue for 50 jobs a minute", "design", "cheap"),
 # docs
 ("is pydantic v3 out yet and what broke", "docs", "cheap"),
 ("latest next.js version and does app router still need use client", "docs", "cheap"),
 ("does python 3.14 remove the gil by default", "docs", "cheap"),
 ("is create-react-app still maintained or dead", "docs", "cheap"),
 ("whats new in tailwind 4", "docs", "cheap"),
 ("any breaking changes going from django 4.2 to 5.1", "docs", "cheap"),
 ("is langchain's old agent api deprecated now", "docs", "cheap"),
 ("current recommended way to do auth in next.js 15", "docs", "cheap"),
 # explain
 ("whats the difference between let and const", "explain", "cheap"),
 ("hey can u explain what a closure is", "explain", "cheap"),
 ("eli5 docker volumes", "explain", "cheap"),
 ("what does git rebase actually do", "explain", "cheap"),
 ("explain in depth how the python gil works, why it exists, how it interacts with io and c extensions, and what free threading changes", "explain", "expensive"),
 ("how does postgres mvcc work under the hood and when does vacuum fall behind", "explain", "expensive"),
 ("what is big o", "explain", "cheap"),
 ("walk me through how react reconciliation decides what to re-render, including keys and the fiber tree", "explain", "expensive"),
 # code
 ("regex for indian phone numbers", "code", "cheap"),
 ("kill process on port 8080", "code", "cheap"),
 ("write a function to check palindrome", "code", "cheap"),
 ("write a python script that watches a folder and uploads new files to s3 with retries, logging and a dry run flag", "code", "expensive"),
 ("sql to get top 3 salaries per department", "code", "cheap"),
 ("make this faster:\n```python\ndef f(a):\n    return [x for x in a if x in a]\n```", "code", "cheap"),
 ("build a cli in go that syncs two directories, supports ignore files, concurrent copying, resume after crash and progress bars", "code", "expensive"),
 ("convert this to typescript\n```js\nfunction add(a, b) { return a + b }\n```", "code", "cheap"),
 # general (non-programming)
 ("capital of australia", "general", "cheap"),
 ("which country is nairobi in", "general", "cheap"),
 ("where is mount kilimanjaro", "general", "cheap"),
 ("who painted the mona lisa", "general", "cheap"),
 ("whats 18 percent of 650", "general", "cheap"),
 ("hi how are you doing today", "general", "cheap"),
 ("write a detailed 1000 word essay comparing the economic causes of the american and haitian revolutions", "general", "expensive"),
 ("plan a 12 day trip through brazil on a budget with transport options, daily costs and backup plans", "general", "expensive"),
 # fresh info (general)
 ("who is the current president of south africa", "docs", "cheap"),
 ("weather in bengaluru today", "docs", "cheap"),
 ("latest news on the chandrayaan mission", "docs", "cheap"),
 # ---- ADD YOUR OWN BELOW, same format ----
]
test_norm = {re.sub(r"\s+", " ", t.lower()).strip() for t, _, _ in MY_TEST}
before = len(df)
df = df[~df.norm.isin(test_norm)].reset_index(drop=True)
print("removed", before - len(df), "training rows that matched a test prompt;", len(MY_TEST), "test prompts")

## 4. Train and evaluate

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from features import featurize, apply_rules, route

bal = pd.concat([g.sample(min(len(g), CAP), random_state=SEED) for _, g in df.groupby(["task","tier"])]).reset_index(drop=True)
print(len(bal), "rows after balancing"); print(bal.groupby(["task","tier"]).size().unstack(fill_value=0))
print(bal.groupby(["source","task"]).size().unstack(fill_value=0))

enc = SentenceTransformer("all-MiniLM-L6-v2")
X = featurize(bal.text.tolist(), enc)
strat = bal.task + "_" + bal.tier
if strat.value_counts().min() < 2: strat = bal.task
idx_tr, idx_te = train_test_split(np.arange(len(bal)), test_size=0.2, stratify=strat, random_state=SEED)
def fit(y, idx): return LogisticRegression(max_iter=3000, C=2.0, class_weight="balanced").fit(X[idx], y.iloc[idx])
task_clf, tier_clf = fit(bal.task, idx_tr), fit(bal.tier, idx_tr)
print("\n=== dataset split (looks optimistic, because prompts resemble each other) ===")
print(classification_report(bal.task.iloc[idx_te], task_clf.predict(X[idx_te])))

In [ ]:
Xm = featurize([m[0] for m in MY_TEST], enc)
pt, pr = task_clf.predict(Xm), tier_clf.predict(Xm)
ok_model = ok_task = ok_tier = ok_route = ok_rule_route = 0
for (t, tk, tr), a, b in zip(MY_TEST, pt, pr):
    a2, b2, why = apply_rules(t, a, b)
    ok_task += a == tk; ok_tier += b == tr
    ok_route += route(a, b) == route(tk, tr)
    ok_rule_route += route(a2, b2) == route(tk, tr)
    flag = "OK  " if (a2 == tk and route(a2, b2) == route(tk, tr)) else "MISS"
    print(flag, f"task {tk}->{a2} | tier {tr}->{b2}", "(rule)" if why else "", "|", repr(t[:60]))
n = len(MY_TEST)
print(f"\nTask accuracy (model only):        {ok_task/n:.2f}")
print(f"Tier accuracy (model only):        {ok_tier/n:.2f}")
print(f"Right MODEL chosen (model only):   {ok_route/n:.2f}   <- the number that matters")
print(f"Right MODEL chosen (with rules):   {ok_rule_route/n:.2f}   <- what the demo will do")
print("Always-expensive baseline would pick the right model for:", round(np.mean([route(tk, tr) == "claude" for _, tk, tr in MY_TEST]), 2))

## 5. Final model, saved with the server files

In [ ]:
import joblib, sklearn, importlib.metadata as md_
task_final = LogisticRegression(max_iter=3000, C=2.0, class_weight="balanced").fit(X, bal.task)
tier_final = LogisticRegression(max_iter=3000, C=2.0, class_weight="balanced").fit(X, bal.tier)
joblib.dump({"task": task_final, "tier": tier_final}, "router.joblib")
with open("requirements.txt", "w") as f:
    f.write(f"scikit-learn=={sklearn.__version__}\nsentence-transformers=={md_.version('sentence-transformers')}\njoblib=={md_.version('joblib')}\nfastapi\nuvicorn\n")
print(open("requirements.txt").read()); print("saved router.joblib")

In [ ]:
%%writefile serve.py
import joblib
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from sentence_transformers import SentenceTransformer
from features import featurize, hand_features, apply_rules

app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
enc = SentenceTransformer("all-MiniLM-L6-v2")
m = joblib.load("router.joblib")

class Req(BaseModel):
    prompt: str

@app.post("/predict")
def predict(r: Req):
    x = featurize([r.prompt], enc)
    tp, rp = m["task"].predict_proba(x)[0], m["tier"].predict_proba(x)[0]
    task, tier = str(m["task"].classes_[tp.argmax()]), str(m["tier"].classes_[rp.argmax()])
    conf = float((tp.max() + rp.max()) / 2)
    task, tier, why = apply_rules(r.prompt, task, tier)
    if why: conf = max(conf, 0.95)
    hf = hand_features(r.prompt)
    if hf[3]: why.append("Contains an error or stack trace")
    if hf[4]: why.append("Mentions versions or recent changes")
    if hf[5]: why.append("Architecture or tradeoff language")
    if hf[2] or hf[1] > 1.5: why.append("Includes pasted code or multiple lines")
    why.append(f"Model view: task {m['task'].classes_[tp.argmax()]} ({tp.max():.0%}), tier {m['tier'].classes_[rp.argmax()]} ({rp.max():.0%})")
    return {"task": task, "tier": tier, "confidence": conf, "reasons": why}

In [ ]:
from google.colab import files
for f in ["router.joblib", "features.py", "serve.py", "requirements.txt"]: files.download(f)

## Run it on your laptop
Put `router.joblib`, `features.py`, `serve.py`, `requirements.txt` in one folder, then:
```
pip install -r requirements.txt
uvicorn serve:app --port 8000
```
Open the updated `prompt-router.html` from your computer (not the published link). If the page shows "Predicted by trained model", it works.

**If a category is weak:** add 50 to 100 messy prompts for it to `my_prompts.csv` (`text,task,tier`), upload to Colab, and Run all again. Do not copy test prompts into it.